# Utilidade - Do desenho ao Gcode

In [1]:
from components import gui
from components import files
from components import path_tools
from components import gcode_tools
from components import images_tools as it
from components import points_tools as pt
from components import morphology_tools as mt
from components import skeleton as skt
from components.timer import Timer
from components.layer import Island, Layer
from components.offset import Offset
import os
import cv2
import numpy as np
import ezdxf
from ezdxf import path as ezdxf_path


def dxf_to_pgm(dxf_path, retx, rety, output_path=None, dpi=300, unit_to_mm=None, margin_px=1):
    """Rasterize DXF geometry inside its outer frame, scaled to retx x rety mm."""
    if dpi <= 0:
        raise ValueError("dpi must be greater than zero")
    if retx <= 0 or rety <= 0:
        raise ValueError("retx and rety must be greater than zero")
    if margin_px < 0:
        raise ValueError("margin_px cannot be negative")

    document = ezdxf.readfile(dxf_path)
    if unit_to_mm is None:
        drawing_units = document.header.get("$INSUNITS", 0)
        unit_to_mm = (
            ezdxf.units.conversion_factor(drawing_units, ezdxf.units.MM)
            if drawing_units
            else 1.0
        )
    if unit_to_mm <= 0:
        raise ValueError("unit_to_mm must be greater than zero")

    paths = []
    flattening_distance = 0.2 * 25.4 / (dpi * unit_to_mm)
    for entity in document.modelspace():
        try:
            entity_path = ezdxf_path.make_path(entity)
            vertices = np.asarray(
                list(entity_path.flattening(distance=flattening_distance, segments=8)),
                dtype=float,
            )
        except (AttributeError, TypeError, ValueError):
            continue
        if len(vertices) >= 2 and np.isfinite(vertices[:, :2]).all():
            paths.append(vertices[:, :2])

    if not paths:
        raise ValueError("The DXF contains no supported drawable modelspace geometry")

    all_points = np.concatenate(paths, axis=0)
    min_x, min_y = all_points.min(axis=0)
    max_x, max_y = all_points.max(axis=0)
    frame_width = max_x - min_x
    frame_height = max_y - min_y
    tolerance = max(frame_width, frame_height, 1.0) * 1e-8
    if frame_width <= tolerance or frame_height <= tolerance:
        raise ValueError("The DXF outer frame must have nonzero width and height")

    frame_edges = {
        "bottom": [],
        "top": [],
        "left": [],
        "right": [],
    }
    frame_entity_indices = set()
    for entity_index, vertices in enumerate(paths):
        for start, end in zip(vertices[:-1], vertices[1:]):
            x1, y1 = start
            x2, y2 = end
            if abs(y1 - min_y) <= tolerance and abs(y2 - min_y) <= tolerance:
                frame_edges["bottom"].append((min(x1, x2), max(x1, x2)))
                frame_entity_indices.add(entity_index)
            elif abs(y1 - max_y) <= tolerance and abs(y2 - max_y) <= tolerance:
                frame_edges["top"].append((min(x1, x2), max(x1, x2)))
                frame_entity_indices.add(entity_index)
            elif abs(x1 - min_x) <= tolerance and abs(x2 - min_x) <= tolerance:
                frame_edges["left"].append((min(y1, y2), max(y1, y2)))
                frame_entity_indices.add(entity_index)
            elif abs(x1 - max_x) <= tolerance and abs(x2 - max_x) <= tolerance:
                frame_edges["right"].append((min(y1, y2), max(y1, y2)))
                frame_entity_indices.add(entity_index)

    def edge_is_complete(intervals, edge_start, edge_end):
        covered_until = edge_start
        for interval_start, interval_end in sorted(intervals):
            if interval_start > covered_until + tolerance:
                return False
            covered_until = max(covered_until, interval_end)
        return covered_until >= edge_end - tolerance

    if not (
        edge_is_complete(frame_edges["bottom"], min_x, max_x)
        and edge_is_complete(frame_edges["top"], min_x, max_x)
        and edge_is_complete(frame_edges["left"], min_y, max_y)
        and edge_is_complete(frame_edges["right"], min_y, max_y)
    ):
        raise ValueError("The DXF must contain an axis-aligned rectangular outer frame")

    drawing_paths = [
        vertices
        for entity_index, vertices in enumerate(paths)
        if entity_index not in frame_entity_indices
    ]
    if not drawing_paths:
        raise ValueError("The DXF outer frame contains no drawable geometry")

    pixels_per_unit_x = retx * dpi / 25.4 / frame_width
    pixels_per_unit_y = rety * dpi / 25.4 / frame_height
    width = int(np.ceil(retx * dpi / 25.4)) + 2 * margin_px + 1
    height = int(np.ceil(rety * dpi / 25.4)) + 2 * margin_px + 1
    image = np.full((height, width), 255, dtype=np.uint8)

    for points in drawing_paths:
        pixels = np.column_stack(
            (
                np.rint((points[:, 0] - min_x) * pixels_per_unit_x + margin_px),
                np.rint((max_y - points[:, 1]) * pixels_per_unit_y + margin_px),
            )
        ).astype(np.int32)
        for start, end in zip(pixels[:-1], pixels[1:]):
            cv2.line(image, tuple(start), tuple(end), 0, 1, lineType=cv2.LINE_8)

    if output_path is None:
        output_path = os.path.splitext(dxf_path)[0] + ".pgm"
    if not cv2.imwrite(output_path, image):
        raise OSError(f"Could not write PGM file: {output_path}")
    return output_path

folders = files.System_Paths(os.getcwd())
path_input, file_name = gui.load_model(folders)
retx, rety = gui.ask_parameters_constructor(
    "Enter the outer frame dimensions in millimeters",
    "DXF frame dimensions",
    ["retx (mm)", "rety (mm)"],
    ["", ""],
)

pgm_input = dxf_to_pgm(path_input, retx=retx, rety=rety, dpi=300, margin_px=1)

gray = cv2.imread(pgm_input, 0)
_, binary = cv2.threshold(gray, 244, 255, cv2.THRESH_BINARY_INV)
img = binary.astype(bool)
img = path_tools.one_pixel_wide(img)
img,a,b = skt.prune(img, min_seg_length=10, iterative_prune=3)

thislayer = Layer(name="L_000", original_img=img, dpi=300)
thislayer.islands = [Island(f"I_000", img)]

w_configurations = files.Config("welding_config.yaml")
[d_tw, frst_d_tw, sob_tw_per, name_prog_t] = gui.ask_parameters_thin_walls(w_configurations)
[void_max,
 external_max,
 internal_max,d_cont, 
 frst_d_cont, 
 sob_cont_per,
 name_prog_c] = gui.ask_parameters_offsets(w_configurations)
[n_max, 
 d_bridg, 
 frst_d_bridg, 
 sob_bridg_per, 
 name_prog_b, 
 connect_offsets] = gui.ask_parameters_bridges(w_configurations)
[d_larg, 
 frst_d_larg, 
 sob_larg_per, 
 name_prog_l, 
 large_weaving_limmit, 
 style] = gui.ask_parameters_bulky(w_configurations)
[vel_vazio, 
 p_entre_int_ext, 
 p_entre_layers,
 substratoy, 
 substratox, 
 cortey, 
 cortex] = gui.ask_parameters_Gcode()
coords_substrato = [substratoy, substratox]
coords_corte = [cortey, cortex]

thislayer.base_frame = img.shape
thislayer.pxl_per_mm = thislayer.dpi / 25.4
thislayer.mm_per_pxl = 1 / thislayer.pxl_per_mm
thislayer.islands[0].contours = Offset("O_000", np.zeros_like(thislayer.original_img), [])
thislayer.islands[0].contours.regions = []
thislayer.program_cont = name_prog_c
thislayer.program_bridg = name_prog_b
thislayer.program_tw = name_prog_t
thislayer.program_larg = name_prog_l
thislayer.void_max = "teste"
thislayer.max_internal_walls = "teste"
thislayer.max_external_walls = "teste"
thislayer.n_max = n_max
thislayer.n_layers = 1
layers = [thislayer]

sequence = path_tools.img_to_chain(thislayer.original_img)
if len(sequence) > 0 and isinstance(sequence[0][0], (list, tuple, np.ndarray)):
    new_seq = []
    for line in sequence:
        # determine a suitable first point for the line using hit-miss ends
        line_img = it.points_to_img(line, np.zeros_like(thislayer.original_img))
        ends = pt.img_to_points(mt.hitmiss_ends_v2(line_img))
        if len(ends) > 0:
            first_pt = ends[0]
            line = path_tools.set_first_pt_in_seq(line, first_pt)
        simplified = path_tools.simplifica_retas_masterV2(
            path_tools.cut_repetition(line), 1, [])
        new_seq.append(simplified)
    
    colored_new_seg = np.zeros_like(thislayer.original_img)
    color = 1
    for route in new_seq:
        colored_new_seg = it.chain_to_lines(route, colored_new_seg.astype(np.uint8),color=color) 
        color += 1
    # a = 8
    # b = 9
    # new_seq[a],new_seq[b] = new_seq[b],new_seq[a]

    # new_seq[9] = new_seq[9][::-1]


    sequence = [pt for line in new_seq for pt in line]
aaaa = it.points_to_img(sequence, np.zeros_like(thislayer.original_img))
layers[0].islands[0].contours.pts_cont = sequence
layers[0].islands[0].island_route = sequence
layers[0].islands[0].thinwalls_tree_route = []
layers[0].islands[0].internal_tree_route = []
layers[0].islands[0].external_tree_route = []


with Timer(f"Step 10R"):
    gcode_tools.layers_to_Gcode_UFSC(layers, 
                                    folders, 
                                    w_configurations,
                                    vel_vazio,
                                    p_entre_int_ext,
                                    p_entre_layers,
                                    coords_substrato,
                                    coords_corte,
                                    flag_drawing=True
                                    )
%reset_selective -f "layers"

Starting -> Step 10R
Switched to 1
Total travel distance 0 = 3508.6779242048174mm
Estimated time with speed 400mm/min = 8.771694810512043min

     Finnished -> Step 10R - Total time: 0.0071849822998046875
